<a href="https://colab.research.google.com/github/chantakorn-uranukul-5792092/chantakorn-uranukul-5792092-AnalyticsEngineering/blob/main/freshcrate_practice_weekly_ops.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analytics Engineering: practice test (set 2)
## Practice: Weekly operations pipeline, FreshCrate

*Modelled on the 5.02 "Plateful" practice test. Same style and marking approach, different company, different data, different traps.*

## How this practice works

This is a rehearsal for the in-class test. It is **not marked** and nothing is submitted. It works only if you treat it like the real thing: attempt it first, and only then look at the model answer.

The conditions match the real test: the same time limit, the same style of notebook, and a problem you have not seen. The data is fake and was generated for this exercise.

- Set a timer for **55 minutes** and work alone. Stop when it rings, then spend **5 minutes** exporting your notebook (see below). Write your start and stop times in the cell under the timing table.
- Work in **Google Colab** (or Jupyter) and save the notebook as `.ipynb`.
- You may use *any materials, the module GitHub code and any generative AI tool*.
- There is no end-point as such. The task is longer than 55 minutes, so you are completing part of a larger job. Do as much as you can, and do it properly.
- The code itself is not marked. Marks in the real test go on how the task was executed, the methodology applied, and the interpretation of results. Practise that.
- In text cells or comments, include:
  1. **how the code was generated**: the prompt you sent, or the source you used;
  2. **evidence of checking**: what you compared, counted or inspected, and what it showed;
  3. **your interpretation in your own words**.

**Tips**

- Run small pieces and know what output you expect before you run them.
- Put enough text next to the code that someone else can follow your reasoning.
- Follow sound analytics engineering method: understand the grain/rows that you are building around, clean in stages, check before you move on.
- Quality over quantity. A smaller pipeline that you have checked beats a larger one you have not.
- Stay calm. Not finishing is expected.

**Suggested timing**

| Minutes | Focus |
| --- | --- |
| 0 to 10 | Explore and profile the tables |
| 10 to 20 | Plan: grain, joins, cleaning decisions |
| 20 to 45 | Build as far as you can |
| 45 to 55 | Check, reconcile and write up |
| 55 to 60 | Export: **Runtime, then Restart session and run all**; File, Download, .ipynb; put the file in a zip |

Keep the exported file so you can compare your work with the model answer afterwards.

**Started at:** ____  **Stopped at:** ____  **Furthest step I reached (see the brief):** ____

## The data

You have six tables from a fictional online grocery service, FreshCrate, which delivers from stores in six UK cities. They cover orders placed from 1 September to 31 December 2025. The tables were extracted on the morning of 2 January 2026.

| Name in this notebook | What it holds |
| --- | --- |
| `orders` | Order records: customer, store, when it was placed, delivery slot type, delivery fee, promotion |
| `order_events` | The order's progress through fulfilment, recorded as events |
| `order_lines` | The products inside each order |
| `payments` | Money movements for each order: charges and refunds |
| `stores` | Store details: name, city and type |
| `customers` | Customer details |

All timestamps in all tables are stored in **UTC**.

**How the tables connect**

- `orders` to `order_events`, `order_lines` and `payments` on `order_id`
- `orders` to `stores` on `store_id`
- `orders` to `customers` on `customer_id`

Check the actual column names in the next cell before you write any joins.

**Getting the data in:** the setup cell asks you to upload `freshcrate-practice.zip` (Colab shows an upload box). If you run Jupyter locally, put the zip in the same folder as the notebook.

In [ ]:
# ---- Setup: run this cell first ----
!pip install -q duckdb

import os
import glob
import zipfile
import shutil
import duckdb
import pandas as pd

DATA_DIR = "freshcrate"
ZIP_NAME = "freshcrate-practice.zip"

FILES = {
    "orders": "orders.csv",
    "order_events": "order_events.csv",
    "order_lines": "order_lines.csv",
    "payments": "payments.csv",
    "stores": "stores.csv",
    "customers": "customers.csv",
}

# If we don't have the csv files already in the data directory:
if not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    # 1. Check if the zip file exists in the current directory or /content
    zip_path = None
    for loc in [ZIP_NAME, os.path.join("/content", ZIP_NAME)]:
        if os.path.exists(loc):
            zip_path = loc
            break

    # 2. If the zip file isn't found, ask the user to upload it
    if not zip_path:
        try:
            from google.colab import files
            print(f"Please upload '{ZIP_NAME}' using the file uploader below:")
            uploaded = files.upload()
            if uploaded:
                zip_path = list(uploaded.keys())[0]
        except ImportError:
            raise FileNotFoundError(f"Could not find '{ZIP_NAME}'. Please upload it to your session.")

    # 3. Extract the zip file
    if zip_path and os.path.exists(zip_path):
        os.makedirs(DATA_DIR, exist_ok=True)
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(DATA_DIR)
        print(f"Successfully extracted {zip_path} into {DATA_DIR}!")

tables, missing = {}, []
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if found:
        tables[name] = pd.read_csv(found[0])
    else:
        missing.append(filename)

if missing:
    found_csvs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
    raise FileNotFoundError(f"Missing files: {missing}. CSV files found under {DATA_DIR}: {found_csvs}.")

# Make every table available as a DuckDB table
con = duckdb.connect()
for name, df in tables.items():
    con.register(name, df)

def q(sql):
    """Run SQL against the tables above and return a pandas DataFrame."""
    return con.sql(sql).df()

for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("   ", list(df.columns))

Please upload 'freshcrate-practice.zip' using the file uploader below:


Saving freshcrate-practice.zip to freshcrate-practice.zip
Successfully extracted freshcrate-practice.zip into freshcrate!
orders: 30,462 rows, 9 columns
    ['order_id', 'customer_id', 'store_id', 'placed_at', 'delivery_slot', 'delivery_fee_gbp', 'promo_code', 'promo_discount_gbp', 'channel']
order_events: 122,642 rows, 5 columns
    ['event_id', 'order_id', 'event_type', 'event_time', 'source']
order_lines: 193,568 rows, 7 columns
    ['line_id', 'order_id', 'product_id', 'category', 'qty_ordered', 'qty_picked', 'unit_price_gbp']
payments: 33,799 rows, 7 columns
    ['payment_id', 'order_id', 'payment_type', 'status', 'amount_gbp', 'paid_at', 'payment_method']
stores: 20 rows, 6 columns
    ['store_id', 'store_name', 'city', 'store_type', 'opened_date', 'updated_at']
customers: 9,052 rows, 6 columns
    ['customer_id', 'email', 'signup_date', 'account_type', 'postcode_area', 'marketing_opt_in']


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Examples: two ways to work with the tables

You can use SQL, pandas, or a mix of both. The next two cells show each, using only the tables and keys described above. Run them and change them to see what they do. You do not have to use them in your answer.

In [ ]:
# Example 1: SQL
# q() runs SQL against any of the tables above and returns a pandas DataFrame.
from IPython.display import display

# Look at the first few rows of a table
display(q("SELECT * FROM orders LIMIT 5"))

# How many rows are there, and how many different order_id values?
display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT order_id) AS n_distinct_orders FROM orders"))

# Join two tables on a key
display(q("""
    SELECT o.order_id, s.city
    FROM orders AS o
    JOIN stores AS s ON o.store_id = s.store_id
    LIMIT 5
"""))

In [ ]:
# Example 2: pandas
# Each table is also a pandas DataFrame, stored in the dictionary `tables`.
from IPython.display import display

orders = tables["orders"]
display(orders.head())        # first few rows
print(orders.shape)           # (number of rows, number of columns)
print(orders.dtypes)          # data type of each column

# The same join as in Example 1, using pandas
joined = tables["orders"].merge(tables["stores"], on="store_id", how="inner")
display(joined[["order_id", "city"]].head())

# You can mix the two: run SQL, then use pandas methods on the result
result = q("SELECT customer_id FROM orders")
print(result["customer_id"].nunique(), "different customer_id values in orders")

## Your brief

**From:** Operations

> We run FreshCrate, an online grocery delivery service with stores in six UK cities. We are starting a weekly operations report and need the first layers of a data pipeline behind it: clean tables first, then one reporting table. For each week, show orders, net revenue and average delivery time by delivery slot (standard or express) and by city. We also need to know how far we can trust the figures.

**What your final table should look like**

One table with columns for week, city, delivery slot, number of orders, net revenue and average delivery time in minutes. State clearly what one row of your table represents.

**Decisions you must make and state**

- Which week does an order belong to?
- Which orders count as orders, and which do not?
- What does "net revenue" mean here, and where does it come from?
- What counts as "delivery time", and for which orders can it be measured?

**A suggested route, in order** (you are not expected to finish)

1. Profile the tables: grain, keys, duplicates, odd values.
2. Build clean tables (a silver layer) with one row per order where that is the grain.
3. Build the one reporting table (gold) from them.
4. Show with checks that the gold table agrees with the source data.
5. If you have time: add cancelled orders and a refund rate to the table, and write automated checks for your clean tables.

## 1. Understand the data

*Look before you build. Write down what you find: what one row of each table represents, how the tables connect, and anything that looks odd or missing.*

Add text here

First we need to look take a glance over all the table to spot any major error or inconsistencies and get a overall look on the type of tables and columns we are working with.

For this I will just call for the heads of the table since there are 6 and there is a time cruch

Prompt: Loop through each of the tables and give their heads with first 5 rows. Show a clear split between each table so that it looks clean.

In [ ]:
for name, df in tables.items():
    print(f"\n{'='*20} {name.upper()} (First 5 Rows) {'='*20}")
    display(df.head(5))
    print(f"Shape: {df.shape[0]:,} rows, {df.shape[1]} columns\n")


==================== ORDERS (First 5 Rows) ====================


,order_id,customer_id,store_id,placed_at,delivery_slot,delivery_fee_gbp,promo_code,promo_discount_gbp,channel
0,FC100000,C03922,S14,2025-09-01 00:16:49,standard,3.49,NaN,0.0,app
1,FC100001,C08992,S12,2025-09-01 00:57:20,standard,3.49,NaN,0.0,app
2,FC100002,C01953,S09,2025-09-01 05:45:31,standard,349.00,NaN,0.0,app
3,FC100003,C01609,S05,2025-09-01 05:49:51,standard,349.00,NaN,0.0,web
4,FC100004,C01431,S13,2025-09-01 07:13:01,express,4.99,NaN,0.0,web


Shape: 30,462 rows, 9 columns


==================== ORDER_EVENTS (First 5 Rows) ====================


,event_id,order_id,event_type,event_time,source
0,E0000001,FC100000,placed,2025-09-01 00:16:49,app
1,E0000002,FC100001,placed,2025-09-01 00:57:20,app
2,E0000003,FC100002,placed,2025-09-01 05:45:33,app
3,E0000004,FC100003,placed,2025-09-01 05:49:51,web
4,E0000005,FC100004,placed,2025-09-01 07:13:03,web


Shape: 122,642 rows, 5 columns


==================== ORDER_LINES (First 5 Rows) ====================


,line_id,order_id,product_id,category,qty_ordered,qty_picked,unit_price_gbp
0,L0000001,FC100000,P0245,Drinks,1,1,1.53
1,L0000002,FC100000,P0232,Chilled,2,2,4.20
2,L0000003,FC100000,P0163,Chilled,2,2,4.26
3,L0000004,FC100000,P0060,Frozen,3,3,2.55
4,L0000005,FC100000,P0254,Fresh produce,1,1,4.94


... (3 outputs left out)

,store_id,store_name,city,store_type,opened_date,updated_at
0,S01,FreshCrate London Central,London,dark_store,2023-08-10,2025-01-15
1,S02,FreshCrate London North,london,dark_store,2022-11-26,2025-01-15
2,S03,FreshCrate London South,LONDON,dark_store,2024-12-23,2025-01-15
3,S04,FreshCrate London East (old),London,partner_shop,2023-05-30,2024-06-01
4,S04,FreshCrate London East,London,partner_shop,2023-05-30,2025-01-15


Shape: 20 rows, 6 columns


==================== CUSTOMERS (First 5 Rows) ====================


,customer_id,email,signup_date,account_type,postcode_area,marketing_opt_in
0,C00001,c00001@mail.example.com,2023-09-28,customer,EC,Y
1,C00002,c00002@mail.example.com,2024-04-25,customer,BS,Y
2,C00003,c00003@mail.example.com,2024-03-22,customer,LS,Y
3,C00004,c00004@mail.example.com,2025-06-16,customer,N,N
4,C00005,c00005@mail.example.com,2023-08-13,customer,SE,N


Shape: 9,052 rows, 6 columns



Okay, I already see some inconsistencies across each table. Now I will move on to cleaning the tables. The data checks and cleaning will go hand in hand

First I will be working with the orders table. I will do the NA value check first and then check for duplicates in the primary key column (order_id) and then pure row duplicates.

Prompt: Loop through the rows of the orders table to check for the NA values

In [ ]:
# Calculate and display the count and percentage of missing (NA) values in the orders table
orders_df = tables['orders']

null_summary = pd.DataFrame({
    'Missing Values (Count)': orders_df.isnull().sum(),

})

print("=== Missing (NA) Values in orders Table ===")
display(null_summary)

=== Missing (NA) Values in orders Table ===


,Missing Values (Count)
order_id,0
customer_id,0
store_id,0
placed_at,0
delivery_slot,0
delivery_fee_gbp,0
promo_code,28753
promo_discount_gbp,0
channel,0


Okay so we have NA values in the promo column. We will let them be since not all orders will have promo

In [ ]:
# 1. Check for duplicate order_id (Primary Key) values
total_orders = len(orders_df)
unique_order_ids = orders_df['order_id'].nunique()
duplicate_pks = total_orders - unique_order_ids

# 2. Check for exact duplicate rows across all columns
duplicate_rows = orders_df.duplicated().sum()

print("=== Duplicate Checks for orders Table ===")
print(f"Total Rows: {total_orders:,}")
print(f"Unique order_id: {unique_order_ids:,}")
print(f"Duplicate order_id count: {duplicate_pks:,}")
print(f"Pure duplicate rows count: {duplicate_rows:,}")

=== Duplicate Checks for orders Table ===
Total Rows: 30,462
Unique order_id: 30,101
Duplicate order_id count: 361
Pure duplicate rows count: 361


I see 361 duplicates in order_id and full row. We need to confirm whether these both values refer to the same rows before removing them

Prompt: From the duplicates output check if they duplicates detected under both the feilds refer to the same rows. I am expecting an check system and tells whether it passed or failed.

In [ ]:
# ---- Check System: Verify if duplicate primary keys are identical rows ----

# Get all rows that have a duplicated order_id
dup_pks_mask = orders_df.duplicated(subset=['order_id'], keep=False)
dup_pks_df = orders_df[dup_pks_mask]

# Get all rows that are completely identical (all columns match)
dup_rows_mask = orders_df.duplicated(keep=False)
dup_rows_df = orders_df[dup_rows_mask]

# Check if the set of duplicated order_id rows matches the set of fully duplicated rows
# (Each unique duplicated order_id should have identical rows)
is_identical_duplicates = dup_pks_df.equals(dup_rows_df)

print("=== DUPLICATE VERIFICATION SYSTEM ===")
print(f"Rows with duplicate 'order_id': {len(dup_pks_df)}")
print(f"Rows with fully identical columns: {len(dup_rows_df)}")

if is_identical_duplicates:
    print("\n[PASSED] All duplicated primary keys represent exact identical row duplicates. They can be safely de-duplicated.")
else:
    print("\n[FAILED] Some duplicate 'order_id' rows have different column values! De-duplication requires deeper investigation.")

=== DUPLICATE VERIFICATION SYSTEM ===
Rows with duplicate 'order_id': 722
Rows with fully identical columns: 722

[PASSED] All duplicated primary keys represent exact identical row duplicates. They can be safely de-duplicated.


Okay perfect we can remove these duplicates now. We will build a copy of a data set called the under the silver tags for version control

Prompt: Create a copy of the orders data set called silver_orders where we drop all the duplicates identified. Give me the new and old number of rows to verify if there was no data loss

In [ ]:
# ---- Create Silver Layer: Deduplicate Orders ----
# Remove identical duplicate rows from the orders table
silver_orders = orders_df.drop_duplicates()

# Register the clean table in DuckDB as well
con.register('silver_orders', silver_orders)

print("=== Silver Orders Dataset Created ===")
print(f"Original orders shape: {orders_df.shape}")
print(f"Silver orders shape: {silver_orders.shape}")
print(f"Removed rows: {len(orders_df) - len(silver_orders)}")

=== Silver Orders Dataset Created ===
Original orders shape: (30462, 9)
Silver orders shape: (30101, 9)
Removed rows: 361


Now we can check for the categorical data to ensure consistent formatting. Here we need to do the check in delivery_slot and channel columns.

Prompt: List the unique values within the delivery_slot and channel columns. I am expecting a small table for both with number of instances they appeared.

In [ ]:
# ---- Check Unique Values & Frequencies: delivery_slot and channel ----
import pandas as pd
from IPython.display import display, HTML

# Calculate value counts for delivery_slot
slot_counts = silver_orders['delivery_slot'].value_counts(dropna=False).reset_index()
slot_counts.columns = ['Delivery Slot Value', 'Count']

# Calculate value counts for channel
channel_counts = silver_orders['channel'].value_counts(dropna=False).reset_index()
channel_counts.columns = ['Channel Value', 'Count']

# Display both summaries
print("=== Frequency Table for 'delivery_slot' ===")
display(slot_counts)

print("\n=== Frequency Table for 'channel' ===")
display(channel_counts)

=== Frequency Table for 'delivery_slot' ===


,Delivery Slot Value,Count
0,standard,23938
1,express,6163



=== Frequency Table for 'channel' ===


,Channel Value,Count
0,app,21627
1,web,8474


Okay so the channel column is clean, but the deliver slot column needs to be made consistest in terms of capitalization

Prompt: for the delivery_slot column, make the formatting consistent and remove any whitespaces. Each entry can be either standard or express. Also compare the count standard and express in the old table with the silver table. Make the updates in the silver table only.

In [ ]:
# Clean the delivery_slot column in silver_orders cleanly using .loc
silver_orders = silver_orders.copy()
silver_orders.loc[:, 'delivery_slot'] = silver_orders['delivery_slot'].astype(str).str.lower().str.strip()

# Re-register the updated silver_orders inside DuckDB
con.register('silver_orders', silver_orders)

# Prepare comparative count: Original (dirty) table
orig_cleaned_slot = orders_df['delivery_slot'].astype(str).str.lower().str.strip()
original_counts = orig_cleaned_slot.value_counts(dropna=False).rename('Original (Standardized Count)')

# Prepare comparative count: Silver table
silver_counts = silver_orders['delivery_slot'].value_counts(dropna=False).rename('Silver (Standardized Count)')

# Combine counts to compare
comparison_df = pd.concat([original_counts, silver_counts], axis=1).fillna(0).astype(int)
comparison_df['Difference (Duplicate Rows Dropped)'] = comparison_df['Original (Standardized Count)'] - comparison_df['Silver (Standardized Count)']

print("=== Delivery Slot Standardization Summary ===")
display(comparison_df)

=== Delivery Slot Standardization Summary ===


,Original (Standardized Count),Silver (Standardized Count),Difference (Duplicate Rows Dropped)
delivery_slot,,,
standard,24212,23938,274
express,6250,6163,87


Okay we still have 30101 entries. Hence there was no data loss. We do however need to check if the standard and express were marked correctly to each row

Prompt: Check if the number of standard and express is correct in the silver data set. What I am expecting is that you create a new copy of the original data set, remove the duplicates and perform the consistent formatting code on it to check if the numbers are correct. Give the answer whether it failed or passed

In [ ]:
# ---- Validation Check: Re-run process on a fresh copy to verify counts ----

# 1. Take a fresh copy of the original dataset
validation_df = orders_df.copy()

# 2. De-duplicate identical rows
validation_df = validation_df.drop_duplicates()

# 3. Apply the formatting to delivery_slot independently
validation_df['delivery_slot'] = validation_df['delivery_slot'].astype(str).str.lower().str.strip()

# 4. Get counts from both DataFrames
silver_slot_counts = silver_orders['delivery_slot'].value_counts(dropna=False)
validation_slot_counts = validation_df['delivery_slot'].value_counts(dropna=False)

# 5. Compare the Series
check_passed = silver_slot_counts.equals(validation_slot_counts)

print("=== INDEPENDENT RECONCILIATION CHECK ===")
print("Current silver_orders counts:")
print(silver_slot_counts)
print("\nValidation copy counts:")
print(validation_slot_counts)

if check_passed:
    print("\n[PASSED] The counts match perfectly. Standard and express records are correctly aligned and cleaned.")
else:
    print("\n[FAILED] The counts do not match! There is a discrepancy between the silver table and the validation copy.")

=== INDEPENDENT RECONCILIATION CHECK ===
Current silver_orders counts:
delivery_slot
standard    23938
express      6163
Name: count, dtype: int64

Validation copy counts:
delivery_slot
standard    23938
express      6163
Name: count, dtype: int64

[PASSED] The counts match perfectly. Standard and express records are correctly aligned and cleaned.


Okay so it was done correctly. Now we can move to ensuring that the date-time formatting in the placed_at column is consistent.

Prompt: Check the data-time formatting in the placed_At column. The format should be like this yyyy-mm-dd hh:mm:ss in a 24hour format. What I am expecting you to do is looping through each row and displaying any row that doesnt pass the check

In [ ]:
import re

# Regex pattern to match yyyy-mm-dd hh:mm:ss in 24-hour format
# Year: 4 digits, Month: 01-12, Day: 01-31
# Hour: 00-23, Minute: 00-59, Second: 00-59
datetime_regex = re.compile(r'^\d{4}-(0[1-9]|1[0-2])-(0[1-9]|[12]\d|3[01]) (0\d|1\d|2[0-3]):[0-5]\d:[0-5]\d$')

invalid_rows = []

# Loop through each row in silver_orders and check 'placed_at'
for index, row in silver_orders.iterrows():
    val = str(row['placed_at'])
    if not datetime_regex.match(val):
        invalid_rows.append({
            'index': index,
            'order_id': row['order_id'],
            'placed_at': val
        })

print("=== DATE-TIME FORMAT VALIDATION ===")
if len(invalid_rows) == 0:
    print("[PASSED] All rows adhere perfectly to the 'yyyy-mm-dd hh:mm:ss' 24-hour format!")
else:
    print(f"[FAILED] Found {len(invalid_rows)} mismatched rows:")
    invalid_df = pd.DataFrame(invalid_rows)
    display(invalid_df)

=== DATE-TIME FORMAT VALIDATION ===
[PASSED] All rows adhere perfectly to the 'yyyy-mm-dd hh:mm:ss' 24-hour format!


Okay all the rows are consistent. Now lastly we need to check the delivery fee and promo discount column for any major ouliers.

Prompt: check the delivery fee and promo discount fee for any major outliers. What I am expecting is using IQR method to find any statistcal outliers. List the rows that have outliers

In [ ]:
def find_iqr_outliers(df, column):
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

# Identify outliers for delivery fee
delivery_outliers, dev_lower, dev_upper = find_iqr_outliers(silver_orders, 'delivery_fee_gbp')
# Identify outliers for promo discount
promo_outliers, promo_lower, promo_upper = find_iqr_outliers(silver_orders, 'promo_discount_gbp')

print(f"=== Delivery Fee GBP Outliers (Bounds: {dev_lower:.2f} to {dev_upper:.2f}) ===")
print(f"Total outliers found: {len(delivery_outliers)}")
if len(delivery_outliers) > 0:
    display(delivery_outliers.head(10))

print(f"\n=== Promo Discount GBP Outliers (Bounds: {promo_lower:.2f} to {promo_upper:.2f}) ===")
print(f"Total outliers found: {len(promo_outliers)}")
if len(promo_outliers) > 0:
    display(promo_outliers.head(10))

=== Delivery Fee GBP Outliers (Bounds: -5.24 to 8.73) ===
Total outliers found: 634


,order_id,customer_id,store_id,placed_at,delivery_slot,delivery_fee_gbp,promo_code,promo_discount_gbp,channel
2,FC100002,C01953,S09,2025-09-01 05:45:31,standard,349.0,NaN,0.0,app
3,FC100003,C01609,S05,2025-09-01 05:49:51,standard,349.0,NaN,0.0,web
148,FC100145,C01590,S10,2025-09-01 20:48:35,standard,349.0,NaN,0.0,app
157,FC100154,C03350,S12,2025-09-01 21:44:57,express,499.0,NaN,0.0,app
269,FC100266,C08042,S02,2025-09-02 15:59:39,standard,349.0,NaN,0.0,app
363,FC100359,C00876,S03,2025-09-03 07:21:46,standard,349.0,NaN,0.0,web
370,FC100366,C03366,S01,2025-09-03 09:05:16,standard,349.0,NaN,0.0,app
428,FC100424,C07597,S02,2025-09-03 13:25:45,express,499.0,NaN,0.0,app
460,FC100455,C06360,S01,2025-09-03 16:38:44,standard,349.0,NaN,0.0,web
467,FC100462,C07419,S18,2025-09-03 17:12:09,standard,349.0,NaN,0.0,app



=== Promo Discount GBP Outliers (Bounds: 0.00 to 0.00) ===
Total outliers found: 1688


,order_id,customer_id,store_id,placed_at,delivery_slot,delivery_fee_gbp,promo_code,promo_discount_gbp,channel
54,FC100053,C04565,S11,2025-09-01 12:25:54,standard,0.00,WELCOME5,5.0,app
59,FC100058,C05259,S14,2025-09-01 12:39:20,standard,0.00,WELCOME5,5.0,web
67,FC100066,C05245,S04,2025-09-01 13:39:10,standard,3.49,WELCOME5,5.0,app
75,FC100074,C02807,S09,2025-09-01 14:49:38,express,4.99,WELCOME5,5.0,web
81,FC100080,C07058,S02,2025-09-01 15:43:50,express,4.99,WELCOME5,5.0,app
98,FC100096,C01247,S01,2025-09-01 17:02:46,standard,3.49,WELCOME5,5.0,app
120,FC100118,C04585,S09,2025-09-01 18:14:22,standard,0.00,WELCOME5,5.0,app
163,FC100160,C07869,S16,2025-09-01 22:48:30,standard,0.00,WELCOME5,5.0,web
234,FC100231,C01465,S01,2025-09-02 12:22:23,standard,3.49,WELCOME5,5.0,app
248,FC100245,C02459,S07,2025-09-02 13:18:39,express,4.99,WELCOME5,5.0,app


Okay considering that promo disocunt was only applicable to certain orders, the rows in the output are logically possible hence they dont need to be removed.

For the ouliers in the delivery fee column, they are concerning becuase delivery fee of more that 100 gbp doesnt make sense. Hence should be excluded from the data set when moving towards the making the gold level aggregate table. I am keeping it in for now considering we might need these row for analysis that doesnt concern deliver fee.

## 2. Plan

*Before you write the main code, write your plan in your own words: what one row of your final table is, which tables you will join and in what order, how many rows you expect at each step, and how you will handle the problems you found.*

Add text here

## 3. Build

*Build as far as you can in the time. Short, re-runnable cells are better than one long cell.*

Add text here as you go along. Show that you are staying in control of the build!

In [ ]:
# add code here

## 4. Check

*Show how you know your answer is right. What did you compare, count or spot check, and what did the checks show? If a check failed, say what you changed.*

Add text here as you go along. Show that you are staying in control of the build!

All the checks were performed while cleaning the data sets. They are done in the steps above

In [ ]:
# add code here

## 5. Prompts and AI notes

*Paste each prompt you used, as you sent it. Then note what you accepted, what you changed or rejected, and why. Include at least one example where the AI was wrong or needed correcting, if that happened.*

The prompts used are mentioned before the code. They are marked with "Prompt:"

## 6. What I could not verify

*Be honest about what remains unchecked, and what you would do next with more time.*

Considering the time crunch I was only able to fully clean the order data set. With more time I would've:

- cleaned the order_event data set. Here I would have checked for any NA values and duplicates in the primary key and full rows. I would have done the categorical check for the event_type and source columns. Lastly I would have ensured consistent formatting for the event_time column. I would also keep the cancelled orders for now incase we need for future aggregation.
- For the order_lines table, I would have done a similar  duplication and NA value check. I wouldve also ensured the consistency in the category column and checked for any major outliers using the IQR method in the numberical data columns.
- For the customer data set I would have done a similar  duplication and NA value check. Ensured no email id duoplicates. Consistency in the data-time column. Consistent formatting in the account_type postcode_area and marketing columns.
- for rest two, payments and stores tables I would have performed similar data quality checks and cleaned them.

All these tables would have been copied to make a silver level cleaned tables.

Next, to get the aggregated gold level data we would need to first join the tables using the primay keys. Next, the order that were placed and deliverd. I would reduce the data set by removing the cancelled orders.


To get rhe duration we will need to subtract the time at which the delivery was done by when the order was placed. The revenue can be calculated by adding the order value with delivery fees and subtracting the promo if any. I woudl remove the delivery fee outliers since they were not plasible. I would then get the aggregation of these numbers by city and and delivery slot.



## Self-check before you stop

Tick these off in your head or in a text cell. They mirror the three things the real test is marked on.

**How was the code generated?**
- [ ] Every block of code that matters has a prompt or a source associated with it.
- [ ] I can say what each block is meant to do and what output I expected.

**Evidence of checking**
- [ ] For every table I built, I checked the number of rows against what I expected, and I can say why.
- [ ] I checked that keys I treat as unique are unique, and I looked for duplicates.
- [ ] My final table agrees with the source data in at least one total, and I show the comparison.
- [ ] I looked at individual records, not only at totals.
- [ ] I wrote down at least one thing the checks caught, and what I changed (if possible).

**Interpretation in my own words**
- [ ] I state each decision (week, which orders count, net revenue, delivery time) and why I chose it.
- [ ] I say what the figures can and cannot be trusted for.
- [ ] Section 6 lists what I did not verify, honestly.